# Initialization

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

from skopt import gp_minimize
from skopt.space import Real
from skopt.learning import GaussianProcessRegressor
from skopt.learning.gaussian_process.kernels import Matern, RBF
from sklearn.preprocessing import OneHotEncoder

import numpy as np
import pandas as pd
from pathlib import Path

## Directories

In [2]:
data_dir = Path("data")
Path.mkdir(data_dir, exist_ok=True)

plot_dir = Path("plots")
Path.mkdir(plot_dir, exist_ok=True)

log_dir = Path("logs")
Path.mkdir(log_dir, exist_ok=True)

## Data reading

In [166]:
data_file = "Dataset_SL.xlsx"

In [167]:
# 1. Define the experimental data
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")

In [168]:
# Encode categorical salt types numerically
# label_encoder = LabelEncoder()
# experiment_data["salt_encoded"] = label_encoder.fit_transform(experiment_data["salt"])  # noqa
salt_encoder = OneHotEncoder(sparse_output=False)
encoded_array = list(
    salt_encoder.fit_transform(np.array(experiment_data["salt"]).reshape(-1, 1))  # noqa
)

# Convert to DataFrame with column names
encoded_df = pd.DataFrame(
    encoded_array, columns=salt_encoder.get_feature_names_out(["salt"])
)

# Concatenate the new encoded columns
experiment_data = pd.concat([experiment_data, encoded_df], axis=1)

In [169]:
# make the other columns as floats
experiment_data = experiment_data.astype(
    {
        "additive_concentration": float,
        "water_to_binder_ratio": float,
        "antisettling_concentration": float,
        "E_d": float,
        "KPI": float,
    }
)

# get optimization round for later use in saving
opt_round = np.max(experiment_data["opt_round"])

In [ ]:
experiment_data

# Search space

In [171]:
search_space = [
    # Categorical([
    #     "MgSO4", "CaCl2", "SrBr2", "MgCl2", "CuSO4", "Al2(SO4)3",
    #     "K2CO3", "KAl(SO4)2", "LiCl", "Mg(NO3)2", "Zn(NO3)2"
    #     ],
    #     name="salt"),
    # Real(0, 10, name="salt_encoded"),
    Real(0.1, 0.9, name="additive_concentration"),
    Real(0.7, 1.5, name="water_to_binder_ratio"),
    Real(0.0, 0.5, name="antisettling_concentration"),
    Real(0.0, 1.0, name="salt_Al2(SO4)3"),
    Real(0.0, 1.0, name="salt_CaCl2"),
    Real(0.0, 1.0, name="salt_CuSO4"),
    Real(0.0, 1.0, name="salt_K2CO3"),
    Real(0.0, 1.0, name="salt_KAl(SO4)2"),
    Real(0.0, 1.0, name="salt_LiCl"),
    Real(0.0, 1.0, name="salt_Mg(NO3)2"),
    Real(0.0, 1.0, name="salt_MgCl2"),
    Real(0.0, 1.0, name="salt_MgSO4"),
    Real(0.0, 1.0, name="salt_SrBr2"),
    Real(0.0, 1.0, name="salt_Zn(NO3)2"),
]

# Optimization

In [172]:
# Define input (X) and outputs (y)
X = experiment_data[
    [
        "additive_concentration",
        "water_to_binder_ratio",
        "antisettling_concentration",
        "salt_Al2(SO4)3",
        "salt_CaCl2",
        "salt_CuSO4",
        "salt_K2CO3",
        "salt_KAl(SO4)2",
        "salt_LiCl",
        "salt_Mg(NO3)2",
        "salt_MgCl2",
        "salt_MgSO4",
        "salt_SrBr2",
        "salt_Zn(NO3)2",
    ]
].to_numpy()
y_energy = experiment_data["E_d"].values
y_kpi = experiment_data["KPI"].values

## Gaussian Process

In [173]:
def fit_gp_models(X, y, kernel):
    """Fits Gaussian Process models to the given experimental data."""  # noqa

    gp = GaussianProcessRegressor(
        kernel=kernel, normalize_y=True, n_restarts_optimizer=10
    )

    gp.fit(X, y)

    return gp

### Matern kernel

In [174]:
gp_energy_matern = fit_gp_models(
    X,
    y_energy,
    Matern(length_scale=1.0, length_scale_bounds=(1e-5, 1e5), nu=2.5),  # noqa
)

In [142]:
gp_kpi_matern = fit_gp_models(
    X, y_kpi, Matern(length_scale=1e-2, length_scale_bounds=(1e-5, 1e1), nu=2.5)  # noqa
)

### RBF kernel

In [143]:
gp_energy_rbf = fit_gp_models(
    X,
    y_energy,
    RBF(
        length_scale=1e-2,
        length_scale_bounds=(1e-5, 1e1),
    ),
)

In [144]:
gp_kpi_rbf = fit_gp_models(
    X,
    y_kpi,
    RBF(
        length_scale=1e-2,
        length_scale_bounds=(1e-5, 1e1),
    ),
)

# Bayesian Optimization

In [145]:
def suggest_new_samples(
    gp_model: GaussianProcessRegressor,
    salt_encoder: OneHotEncoder,
    verbose: bool = False,
):
    """Suggests new samples based on the trained GP model using different acquisition functions."""  # noqa

    acq_functions = ["EI", "PI", "LCB_low", "LCB_med", "LCB_high"]
    kappa_values = {"LCB_low": 1.0, "LCB_med": 2.5, "LCB_high": 5.0}
    new_samples = []

    for acquisition in acq_functions:  # Limit to requested number
        res = gp_minimize(
            lambda x: gp_model.predict([x])[0],  # Optimize our surrogate model
            dimensions=search_space,  # pass our search space
            base_estimator=gp_model,  # ask for our custom estimators
            acq_func=(
                "LCB"
                if acquisition in ["LCB_low", "LCB_med", "LCB_high"]
                else acquisition
            ),  # define the acquisition function
            kappa=kappa_values.get(
                acquisition, 2
            ),  # define the custom k value if acquisition if "LCB", otherwise k=2  # noqa
            n_calls=20,
            n_initial_points=10,
            initial_point_generator="lhs",
        )

        # Convert numerical salt encoding back to categorical
        suggested = res.x
        if verbose:
            print("Before reverse encoding:")
            print(suggested)

        to_invert = np.array(suggested[3:]).reshape(1, -1)
        suggested_sample = list(
            [salt_encoder.inverse_transform(to_invert)]
        )  # reverse encoding

        suggested_sample = suggested_sample[0][0]
        suggested_sample.extend(list(suggested[:3]))
        suggested_sample.append(str(gp_model.kernel).split("(")[0])
        suggested_sample.append(str(acquisition))
        if verbose:
            print("After reverse encoding:")
            print(suggested_sample)
        new_samples.append(suggested_sample)

    columns = [
        "salt",
        "additive_concentration",
        "water_to_binder_ratio",
        "antisettling_concentration",
        "kernel",
        "acquisition",
    ]

    return pd.DataFrame(new_samples, columns=columns)

### Generate 10 new samples for Energy Density optimization

In [ ]:
# Matérn kernel
new_samples_energy_matern = suggest_new_samples(
    gp_energy_matern, salt_encoder, verbose=True
)

In [147]:
# RBF kernel
new_samples_energy_rbf = suggest_new_samples(gp_energy_rbf, salt_encoder)

In [ ]:
new_samples_energy_matern

In [ ]:
new_samples_energy_rbf

### Generate 10 new samples for economic KPI optimization

In [151]:
# Matérn kernel
new_samples_kpi_matern = suggest_new_samples(gp_kpi_matern, salt_encoder)

In [152]:
# RBF kernel
new_samples_kpi_rbf = suggest_new_samples(gp_kpi_rbf, salt_encoder)

## Saving the new suggestions in the original excel sheet

In [ ]:
# Combine sets of new samples
new_samples = pd.concat(
    [
        new_samples_energy_matern,
        new_samples_energy_rbf,
        new_samples_kpi_matern,
        new_samples_kpi_rbf,
    ],
    ignore_index=True,
)

new_samples.insert(0, "opt_round", opt_round + 1)  # add optimization round

# Read the existing Excel sheet into a DataFrame
experiment_data = pd.read_excel(data_dir / data_file, sheet_name="Datasheet")
# Append the new data to the existing DataFrame
combined_data = pd.concat(
    [experiment_data, new_samples], ignore_index=True, axis=0
)  # noqa

# Write the updated DataFrame back to the same Excel sheet
with pd.ExcelWriter(
    data_dir / data_file, engine="openpyxl", mode="a", if_sheet_exists="replace"  # noqa
) as writer:
    combined_data.to_excel(writer, sheet_name="Datasheet", index=False)

print(
    "New batch of 20 samples suggested. Please conduct experiments and update the dataset."  # noqa
)